In [44]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [45]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    sr: float
    boundaryperball: float
    boundarypercentage: float
    summary: str


In [46]:
def calculate_sr(state: BatsmanState):
    runs = state["runs"]
    balls = state["balls"]

    sr = (runs / balls) * 100

    return {
        "sr": round(sr, 2)
    }


def calculate_bpb(state: BatsmanState):
    balls = state["balls"]
    fours = state["fours"]
    sixes = state["sixes"]

    total_boundaries = fours + sixes

    bpb = total_boundaries / balls

    return {
        "boundaryperball": round(bpb, 2)
    }


def calculate_boundary_percentage(state: BatsmanState):
    runs = state["runs"]
    fours = state["fours"]
    sixes = state["sixes"]

    boundary_runs = (fours * 4) + (sixes * 6)

    percentage = (boundary_runs / runs) * 100

    return {
        "boundarypercentage": round(percentage, 2)
    }


def summary(state: BatsmanState):
    summary_text = (
        f"The batsman scored {state['runs']} runs in "
        f"{state['balls']} balls with a strike rate of "
        f"{state['sr']}. "
        f"{state['boundarypercentage']}% of his runs came "
        f"from boundaries. He hit {state['fours']} fours "
        f"and {state['sixes']} sixes."
    )

    return {
        "summary": summary_text
    }

In [47]:
graph = StateGraph(BatsmanState)

graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
graph.add_node("summary", summary)

# Parallel Execution
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percentage")

graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percentage", "summary")

graph.add_edge("summary", END)

workflow = graph.compile()

In [49]:
result = workflow.invoke({
    "runs": 84,
    "balls": 52,
    "fours": 8,
    "sixes": 4
})

print(result)
print("\n\n\n")
print(result["summary"])

{'runs': 84, 'balls': 52, 'fours': 8, 'sixes': 4, 'sr': 161.54, 'boundaryperball': 0.23, 'boundarypercentage': 66.67, 'summary': 'The batsman scored 84 runs in 52 balls with a strike rate of 161.54. 66.67% of his runs came from boundaries. He hit 8 fours and 4 sixes.'}




The batsman scored 84 runs in 52 balls with a strike rate of 161.54. 66.67% of his runs came from boundaries. He hit 8 fours and 4 sixes.
